In [6]:
# Install Dassl
!git clone https://github.com/KaiyangZhou/Dassl.pytorch.git
%cd Dassl.pytorch/
!pip install -r requirements.txt
!python setup.py install
%cd ..

Cloning into 'Dassl.pytorch'...
remote: Enumerating objects: 2477, done.
remote: Counting objects: 100% (1081/1081), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 2477 (delta 933), reused 846 (delta 846), pack-reused 1396 (from 1)
Receiving objects: 100% (2477/2477), 410.19 KiB | 2.21 MiB/s, done.
Resolving deltas: 100% (1676/1676), done.
/content/CeKALA/Dassl.pytorch
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.9/69.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.3/185.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 76.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https

# CeKALA Experiment Notebook
This notebook allows you to easily run the Multi-Modal Adapter (MMA) experiments on Google Colab.

In [1]:
import os

In [2]:
# 2. Git setup and repository update
repo_url = "https://github.com/tomal66/CeKALA.git"
repo_dir = "CeKALA"
branch = "main"

if not os.path.exists(repo_dir):
    !git clone {repo_url}
    %cd {repo_dir}
    !git checkout {branch}
else:
    %cd {repo_dir}
    !git fetch origin {branch}
    !git reset --hard origin/{branch}

%pip install -r requirements.txt

/content/CeKALA
From https://github.com/tomal66/CeKALA
 * branch            main       -> FETCH_HEAD
HEAD is now at 0a6aad2 feat: Add CeKALA dynamic layer selection algorithm
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dassl-0.6.3-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [3]:
# 3. Download Dataset (Caltech101 Example)
import os
import torchvision
if not os.path.exists("data/caltech-101/101_ObjectCategories"):
    !mkdir -p data/caltech-101
    print("Downloading Caltech101 using torchvision...")
    torchvision.datasets.Caltech101(root="data/caltech-101", download=True)
    # torchvision extracts to data/caltech-101/caltech101/101_ObjectCategories. We move it to expected path:
    if os.path.exists("data/caltech-101/caltech101/101_ObjectCategories"):
        !mv data/caltech-101/caltech101/101_ObjectCategories data/caltech-101/
        !rm -rf data/caltech-101/caltech101
if not os.path.exists("data/caltech-101/split_zhou_Caltech101.json"):
    print("Downloading split file...")
    !gdown 1hyarUivQE36mY6jSomru6Fjd-JzwcCzN -O data/caltech-101/split_zhou_Caltech101.json


In [4]:
# 4. Hyperparameters
DATASET = "caltech101"
SEED = 1
CFG = "vit_b16_ep5"
SHOTS = 16
DATA = "data/"
TRAINER = "MultiModalAdapter"
LOADEP = 5
SUB = "new"

In [5]:
# 5. Imports
import sys
import torch

from dassl.utils import setup_logger, set_random_seed, collect_env_info
from dassl.engine import build_trainer

# Custom modules from CeKALA
import datasets.oxford_pets
import datasets.oxford_flowers
import datasets.fgvc_aircraft
import datasets.dtd
import datasets.eurosat
import datasets.stanford_cars
import datasets.food101
import datasets.sun397
import datasets.caltech101
import datasets.ucf101
import datasets.imagenet

import datasets.imagenet_sketch
import datasets.imagenetv2
import datasets.imagenet_a
import datasets.imagenet_r

import trainers.mmadapter
from train import setup_cfg, print_args

In [6]:
# 6. Setup Arguments and Config for Base Training
class Args:
    pass

args_train = Args()
args_train.root = DATA
args_train.output_dir = f"output/base2new/train_base/{DATASET}/shots_{SHOTS}/{TRAINER}/seed{SEED}"
args_train.resume = ""
args_train.seed = SEED
args_train.source_domains = None
args_train.target_domains = None
args_train.transforms = None
args_train.config_file = f"configs/trainers/{TRAINER}/{CFG}.yaml"
args_train.dataset_config_file = f"configs/datasets/{DATASET}.yaml"
args_train.trainer = TRAINER
args_train.backbone = ""
args_train.head = ""
args_train.eval_only = False
args_train.model_dir = ""
args_train.load_epoch = None
args_train.no_train = False
args_train.opts = ["DATASET.NUM_SHOTS", str(SHOTS), "DATASET.SUBSAMPLE_CLASSES", "base"]

cfg_train = setup_cfg(args_train)
if cfg_train.SEED >= 0:
    print(f"Setting fixed seed: {cfg_train.SEED}")
    set_random_seed(cfg_train.SEED)

setup_logger(args_train.output_dir)

if torch.cuda.is_available() and cfg_train.USE_CUDA:
    torch.backends.cudnn.benchmark = True

print_args(args_train, cfg_train)
print("Collecting env info ...")
print("** System info **\n{}\n".format(collect_env_info()))

print("Building trainer...")
trainer = build_trainer(cfg_train)

Setting fixed seed: 1
***************
** Arguments **
***************
backbone: 
config_file: configs/trainers/MultiModalAdapter/vit_b16_ep5.yaml
dataset_config_file: configs/datasets/caltech101.yaml
eval_only: False
head: 
load_epoch: None
model_dir: 
no_train: False
opts: ['DATASET.NUM_SHOTS', '16', 'DATASET.SUBSAMPLE_CLASSES', 'base']
output_dir: output/base2new/train_base/caltech101/shots_16/MultiModalAdapter/seed1
resume: 
root: data/
seed: 1
source_domains: None
target_domains: None
trainer: MultiModalAdapter
transforms: None
************
** Config **
************
DATALOADER:
  K_TRANSFORMS: 1
  NUM_WORKERS: 8
  RETURN_IMG0: False
  TEST:
    BATCH_SIZE: 100
    SAMPLER: SequentialSampler
  TRAIN_U:
    BATCH_SIZE: 32
    N_DOMAIN: 0
    N_INS: 16
    SAME_AS_X: True
    SAMPLER: RandomSampler
  TRAIN_X:
    BATCH_SIZE: 16
    N_DOMAIN: 0
    N_INS: 16
    SAMPLER: RandomSampler
DATASET:
  ALL_AS_UNLABELED: False
  CIFAR_C_LEVEL: 1
  CIFAR_C_TYPE: 
  NAME: Caltech101
  NUM_LABELE

100%|███████████████████████████████████████| 351M/351M [00:04<00:00, 71.2MiB/s]


Building custom CLIP
Turning off gradients in both the image and the text encoder
Parameters to be updated: {'adapter_learner.text_adapter.11.up.weight', 'adapter_learner.visual_adapter.11.down.0.weight', 'adapter_learner.visual_adapter.8.up.bias', 'adapter_learner.text_adapter.10.down.0.bias', 'adapter_learner.visual_adapter.7.down.0.bias', 'adapter_learner.shared_adapter.8.0.weight', 'adapter_learner.visual_adapter.12.down.0.bias', 'adapter_learner.text_adapter.5.down.0.bias', 'adapter_learner.text_adapter.8.up.weight', 'adapter_learner.text_adapter.11.down.0.weight', 'adapter_learner.visual_adapter.7.down.0.weight', 'adapter_learner.visual_adapter.7.up.bias', 'adapter_learner.text_adapter.7.down.0.weight', 'adapter_learner.text_adapter.10.up.bias', 'adapter_learner.shared_adapter.11.0.weight', 'adapter_learner.shared_adapter.5.0.weight', 'adapter_learner.text_adapter.7.up.bias', 'adapter_learner.visual_adapter.8.up.weight', 'adapter_learner.shared_adapter.11.0.bias', 'adapter_learne

/content/CeKALA/trainers/mmadapter.py:260: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if cfg.TRAINER.MMADAPTER.PREC == "amp" else None


Running CeKALA layer selection on a small subset...
Running CeKALA layer selection for caltech101 with 16 shots...
Algorithm selected layers: [6, 8, 10, 11]
Rebuilding trainer with selected layers...
Loading trainer: MultiModalAdapter
Loading dataset: Caltech101
Reading split from /content/CeKALA/data/caltech-101/split_zhou_Caltech101.json
Loading preprocessed few-shot data from /content/CeKALA/data/caltech-101/split_fewshot/shot_16-seed_1.pkl
SUBSAMPLE BASE CLASSES!
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.08, 1.0))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  ----------
Dataset    Caltech101
# classes  50
# train_x  800
# val   

In [ ]:
# 6.5 Run CeKALA Layer Selection Algorithm
import sys
if '.' not in sys.path:
    sys.path.append('.') # Ensure root is in path
from algorithms.CeKALA import select_layers

print("Running CeKALA layer selection on a small subset...")
selected_layers, _, _ = select_layers(DATASET, SHOTS, cfg_train, k=4)
print(f"Algorithm selected layers: {selected_layers}")

# Update the config manually
cfg_train.defrost()
cfg_train.TRAINER.MMADAPTER.SELECTED_LAYERS = selected_layers
cfg_train.freeze()

print("Rebuilding trainer with selected layers...")
trainer = build_trainer(cfg_train)


In [8]:
# 7. Start Training on Base Classes
print("Starting training on base classes...")
trainer.train()

/content/CeKALA/trainers/mmadapter.py:273: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
100%|██████████| 16/16 [00:09<00:00,  1.61it/s]


In [9]:
# 8. Setup Arguments and Config for New Classes Testing
args_test = Args()
args_test.root = DATA
args_test.output_dir = f"output/base2new/test_{SUB}/{DATASET}/shots_{SHOTS}/{TRAINER}/seed{SEED}"
args_test.resume = ""
args_test.seed = SEED
args_test.source_domains = None
args_test.target_domains = None
args_test.transforms = None
args_test.config_file = f"configs/trainers/{TRAINER}/{CFG}.yaml"
args_test.dataset_config_file = f"configs/datasets/{DATASET}.yaml"
args_test.trainer = TRAINER
args_test.backbone = ""
args_test.head = ""
args_test.eval_only = True
args_test.model_dir = args_train.output_dir
args_test.load_epoch = LOADEP
args_test.no_train = True
args_test.opts = ["DATASET.NUM_SHOTS", str(SHOTS), "DATASET.SUBSAMPLE_CLASSES", SUB]

cfg_test = setup_cfg(args_test)
if cfg_test.SEED >= 0:
    set_random_seed(cfg_test.SEED)
setup_logger(args_test.output_dir)

# Apply dynamic layer selection to test config
cfg_test.defrost()
cfg_test.TRAINER.MMADAPTER.SELECTED_LAYERS = selected_layers
cfg_test.freeze()

trainer_test = build_trainer(cfg_test)
trainer_test.load_model(args_test.model_dir, epoch=args_test.load_epoch)

/content/CeKALA/trainers/mmadapter.py:260: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if cfg.TRAINER.MMADAPTER.PREC == "amp" else None


In [10]:
# 9. Start Evaluation on New Classes
print("Starting evaluation on new classes...")
trainer_test.test()

100%|██████████| 10/10 [00:05<00:00,  1.67it/s]


93.77729257641921